# AI4I 2020: composite failure and mode labels

Evidence class: synthetic structural and label-semantics evidence. This notebook verifies the canonical local CSV through the existing Isoprax verifier and charts only verifier-approved report projections. AI4I's five mode counts can overlap and are not prediction inputs for the composite label. A passing structural check is not predictive efficacy or Semantic/Full Conformance.

Expected local input: `data/ai4i2020.csv`, or set `ISOPRAX_DATA_DIR` before starting Jupyter.

In [ ]:
import os
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
working_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file() and (candidate / "isoprax").is_dir()
    ),
    None,
)
configured_root = os.environ.get("ISOPRAX_REPO_ROOT")
REPO_ROOT = (
    Path(configured_root).expanduser().resolve() if configured_root else working_root
)
if (
    REPO_ROOT is None
    or not (REPO_ROOT / "pyproject.toml").is_file()
    or not (REPO_ROOT / "isoprax").is_dir()
):
    raise RuntimeError(
        "Launch from this checkout or set ISOPRAX_REPO_ROOT to its repository root."
    )
if configured_root and working_root is not None and REPO_ROOT != working_root:
    raise RuntimeError(
        "ISOPRAX_REPO_ROOT does not match the notebook's checkout directory."
    )
repo_path = str(REPO_ROOT)
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0, repo_path)

import isoprax

PACKAGE_ROOT = Path(isoprax.__file__).resolve().parents[1]
if PACKAGE_ROOT != REPO_ROOT:
    raise RuntimeError(
        "The selected Python kernel does not import Isoprax from this checkout."
    )

from notebooks._support import (
    data_root_from_environment,
    display_review,
    review_dataset,
)

DATA_ROOT = data_root_from_environment(REPO_ROOT)
CSV_PATH = DATA_ROOT / "ai4i2020.csv"
print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Local CSV: {CSV_PATH}")

In [ ]:
review = review_dataset("ai4i2020", {"csv": CSV_PATH}, REPO_ROOT)
display_review(review)

The chart projection keeps composite failure, mode labels, rows with multiple modes, and label discrepancies distinct. After verification, the notebook audits label prevalence/overlap and plots univariate sensor distributions. Its fixed comparison is a prior-only classifier versus logistic regression on the earlier 80% of ordered UDI rows, tested on the later 20%; UDI is only an ordering key. Identifiers and all mode labels are excluded as predictors. If verification fails, no row-level analysis is shown.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from isoprax.ai4i2020 import AI4I2020_LABEL_COLUMNS, AI4I2020_MODE_COLUMNS
from notebooks import _analysis

if review.verification.status in {"verified", "verified_with_warnings"}:
    frame = pd.read_csv(CSV_PATH)
    labels = frame.loc[:, AI4I2020_LABEL_COLUMNS]
    label_summary = pd.DataFrame(
        {
            "positive_count": labels.sum().astype(int),
            "prevalence": labels.mean(),
        }
    )
    display(Markdown("## Label audit: composite failure and overlapping modes"))
    display(label_summary)
    frame["reported_mode_count"] = frame.loc[:, AI4I2020_MODE_COLUMNS].sum(axis=1)
    overlap = frame["reported_mode_count"].value_counts().sort_index()
    display(pd.DataFrame({"rows": overlap, "share": overlap / len(frame)}))

    figure, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    label_summary["positive_count"].plot.bar(ax=axes[0], color="#3977a8")
    axes[0].set_title("Observed positive rows by label")
    axes[0].set_ylabel("Rows")
    overlap.plot.bar(ax=axes[1], color="#b87932")
    axes[1].set_title("Rows by number of reported failure modes")
    axes[1].set_xlabel("Reported modes on row")
    axes[1].set_ylabel("Rows")
    display(figure)
    plt.close(figure)

    display(Markdown("## Sensor distributions by composite label"))
    numeric_features = [
        column for column in _analysis.AI4I_MODEL_FEATURES if column != "Type"
    ]
    figure, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    for column, axis in zip(numeric_features, axes.flat):
        frame.boxplot(column=column, by="Machine failure", ax=axis, grid=False)
        axis.set_title(column)
        axis.set_xlabel("Machine failure (0/1)")
    axes.flat[-1].axis("off")
    figure.suptitle("Univariate distributions; descriptive, not causal")
    display(figure)
    plt.close(figure)

    display(Markdown("## Fixed ordered-UDI holdout baseline"))
    train_indices, test_indices = _analysis.ordered_holdout_indices(
        frame["UDI"], train_fraction=0.8
    )
    y = frame["Machine failure"].astype(int)
    train_y = y.iloc[train_indices]
    test_y = y.iloc[test_indices]
    print(
        f"Train: {len(train_indices):,} earlier UDI rows; test: {len(test_indices):,} later UDI rows"
    )
    print(
        f"Train positives: {int(train_y.sum()):,}; test positives: {int(test_y.sum()):,}"
    )
    if train_y.nunique() < 2:
        display(
            Markdown(
                "Baseline not estimable: training partition contains only one outcome class."
            )
        )
    else:
        X = frame.loc[:, _analysis.AI4I_MODEL_FEATURES]
        numeric = [
            column for column in _analysis.AI4I_MODEL_FEATURES if column != "Type"
        ]
        preprocess = ColumnTransformer(
            [
                (
                    "numeric",
                    make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
                    numeric,
                ),
                ("type", OneHotEncoder(handle_unknown="ignore"), ["Type"]),
            ]
        )
        models = {
            "prior dummy": DummyClassifier(strategy="prior"),
            "logistic regression": make_pipeline(
                preprocess, LogisticRegression(max_iter=1000, random_state=41)
            ),
        }
        metric_rows = []
        for name, model in models.items():
            model.fit(X.iloc[train_indices], train_y)
            probabilities = model.predict_proba(X.iloc[test_indices])[:, 1]
            metric_rows.append(
                {
                    "model": name,
                    **_analysis.binary_classification_metrics(test_y, probabilities),
                }
            )
        metrics = pd.DataFrame(metric_rows).set_index("model")
        display(metrics)
        figure, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
        metrics["average_precision"].astype(float).plot.bar(ax=axes[0], color="#3977a8")
        axes[0].set_title("Average precision (test support shown above)")
        metrics["brier_score"].astype(float).plot.bar(ax=axes[1], color="#b87932")
        axes[1].set_title("Brier score (lower is better)")
        display(figure)
        plt.close(figure)
else:
    display(
        Markdown(
            "Exploration is gated: raw rows are read only after the canonical verifier succeeds."
        )
    )